# ReproPilot Hands-On Tutorial
## AI-Assisted Reproducibility Checking for Scientific Software

**Audience:** researchers, research software engineers, faculty, students, and HPC/AI practitioners  
**Level:** introductory to intermediate Python

This self-paced tutorial demonstrates how ReproPilot evaluates a scientific software repository using a deterministic, evidence-based rubric. An optional local language model can explain and prioritize findings, but it never calculates or changes the score.

> **Key idea:** Reproducibility is not a single file or tool. It is a connected practice involving documentation, dependencies, tests, workflows, environments, licensing, provenance, and human review.


## What You Will Learn

By the end of this tutorial, you will be able to:

1. Explain the difference between **deterministic assessment** and **AI-generated explanation**.
2. Run ReproPilot on a scientific software repository.
3. Interpret category scores, findings, and repository evidence.
4. Compare weak and strong reproducibility practices.
5. Identify concrete improvements for an AI, scientific, or HPC project.
6. Discuss the limitations of automated reproducibility assessment.


## Getting Started

Before running the tutorial:

1. Clone the repository and open this notebook from the repository root.
2. Create and activate a Python environment.
3. Install the project and notebook dependencies.
4. Optionally install Ollama for the local-AI demonstration.
5. Run the cells in order.

Suggested terminal commands:

```bash
git clone https://github.com/szuananwar/ai-assisted-reproducibility-bssw.git
cd ai-assisted-reproducibility-bssw

python3 -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate

python -m pip install --upgrade pip
python -m pip install -e ".[dev]"
jupyter lab
```

Optional local AI setup:

```bash
ollama pull gemma3:1b
ollama serve
```


## Conceptual Model

ReproPilot separates two responsibilities:

### 1. Deterministic assessment
The score is computed from explicit checks applied to repository evidence. This makes the result repeatable and inspectable.

### 2. Optional AI explanation
A local language model receives selected evidence and deterministic findings. It can summarize, prioritize, and explain, but it cannot alter the score.

This separation is important because a fluent explanation should never replace traceable evidence.


In [ ]:
from pathlib import Path
import json
import tempfile
import sys

# Make the repository root importable whether Jupyter starts from the repo root
# or from the notebooks/ directory.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "checker").is_dir():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError(
        "Could not locate the repository root. Start Jupyter from the cloned "
        "ai-assisted-reproducibility-bssw repository."
    )

from checker.reproducibility_checker import (
    assess_repository,
    build_evidence_package,
    discover_project_root,
    local_llm_recommendations,
    print_assessment,
)

print("Imports successful.")
print("Python:", sys.version.split()[0])


# Part 1 — Assess a Repository

Begin with the ReproPilot repository so that the demonstration can be reproduced consistently.

The `DOMAIN` value adjusts the assessment context:

- `general`
- `biomedical`
- `climate`
- `hpc-simulation`

### Reflection

Consider which reproducibility concerns are specific to your scientific domain and whether they would be captured by a general software checklist.


In [ ]:
PROJECT_PATH = discover_project_root()
DOMAIN = "general"

print("Project:", PROJECT_PATH)
print("Domain:", DOMAIN)

result = assess_repository(PROJECT_PATH, DOMAIN)
print_assessment(result)


## Understanding the Assessment

Review the assessment and identify:

- the overall score;
- the strongest category;
- the weakest category;
- one passing check;
- one missing or incomplete practice;
- one recommendation that could be implemented immediately.

> **Key point:** A score is a conversation starter, not a certificate of scientific correctness.


In [ ]:
print("Top-level result fields:")
print(list(result.keys()))

print("\nCompact JSON preview:")
print(json.dumps(result, indent=2, default=str)[:4000])


# Part 2 — Inspect the Repository Evidence

A trustworthy assessment should show **why** it reached a conclusion. ReproPilot creates an evidence package containing:

- a file inventory;
- selected file snippets;
- deterministic findings and scores;
- repository context for optional explanation.

### Reflection

What evidence would you require before trusting an automated recommendation?


In [ ]:
evidence = build_evidence_package(PROJECT_PATH, result)

print("Files inventoried:", len(evidence["file_inventory"]))
print("Selected snippets:", list(evidence["selected_file_snippets"]))
print("\nDeterministic score:")
print(json.dumps(evidence["deterministic_score"], indent=2))


In [ ]:
print("Sample file inventory entries:")
for item in evidence["file_inventory"][:15]:
    print(item)


## Interpreting the Evidence

Choose two selected files and consider:

1. What reproducibility claim can this file support?
2. What can it **not** prove?
3. How might a file-presence check create a false sense of confidence?

Examples:

- A `requirements.txt` file supports dependency documentation, but does not guarantee solvability or exact environment recreation.
- A `tests/` directory supports the presence of tests, but does not prove meaningful scientific validation.
- A container recipe supports environment packaging, but does not prove the image builds or produces correct results.


In [ ]:
for filename, snippet in evidence["selected_file_snippets"].items():
    print("=" * 80)
    print(filename)
    print("-" * 80)
    print(str(snippet)[:1200])
    print()


# Part 3 — Optional Privacy-Preserving Local AI Explanation

This section is optional. It requires a local Ollama server and model.

The model receives repository evidence already collected by ReproPilot. It is asked to explain and prioritize deterministic findings. The model does **not** calculate the score.

Skip this cell when Ollama is unavailable. The rest of the tutorial remains fully functional.


In [ ]:
try:
    llm_result = local_llm_recommendations(evidence)
    print(json.dumps(llm_result, indent=2)[:6000])
except Exception as exc:
    print("Local LLM demonstration skipped.")
    print("Reason:", exc)
    print("\nTo enable it, run: ollama pull gemma3:1b && ollama serve")


## Responsible Use of AI

Consider the following questions:

- Why should the AI not control the score?
- What kinds of hallucination could occur when explaining repository evidence?
- What information should remain local or private?
- When is human review mandatory?
- How could the explanation be evaluated for usefulness and faithfulness?

> **Key takeaway:** AI is most defensible here as an explanation layer grounded in visible evidence, not as an untraceable judge.


# Part 4 — Before-and-After Reproducibility Demonstration

This section creates two temporary repositories:

- **Weak repository:** only a minimal README.
- **Stronger repository:** includes common reproducibility and sustainability artifacts.

The comparison shows how visible software practices influence the deterministic assessment.


In [ ]:
def write_file(root: Path, relative_path: str, text: str = "x\n") -> None:
    path = root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)

base = Path(tempfile.mkdtemp(prefix="repropilot_tutorial_"))
weak = base / "weak_project"
strong = base / "strong_project"
weak.mkdir()
strong.mkdir()

write_file(
    weak,
    "README.md",
    "# Minimal Scientific Project\n\nA short description with no setup or testing instructions.\n",
)

strong_files = {
    "README.md": "# Reproducible Scientific Project\n\nIncludes setup, execution, testing, and citation guidance.\n",
    "requirements.txt": "numpy==1.26.4\n",
    "environment.yml": "name: demo\ndependencies:\n  - python=3.11\n  - numpy=1.26.4\n",
    "spack.yaml": "spack:\n  specs: []\n",
    "tests/test_smoke.py": "def test_smoke():\n    assert 2 + 2 == 4\n",
    "apptainer.def": "Bootstrap: docker\nFrom: python:3.11-slim\n",
    "MLproject": "name: reproducible-demo\n",
    "LICENSE": "Example license placeholder for tutorial purposes.\n",
    "CITATION.cff": "cff-version: \"1.2.0\"\ntitle: \"Reproducible Demo\"\n",
    ".github/workflows/ci.yml": "name: CI\non: [push]\njobs: {}\n"
}

for relative_path, text in strong_files.items():
    write_file(strong, relative_path, text)

print("Temporary projects created under:", base)


In [ ]:
weak_result = assess_repository(weak)
strong_result = assess_repository(strong)

print("BEFORE: weak repository")
print_assessment(weak_result)

print("\n" + "=" * 90 + "\n")
print("AFTER: stronger repository")
print_assessment(strong_result)


## Compare the Results

Consider the following questions:

1. Which artifacts improved the score?
2. Which additions improve reproducibility versus maintainability?
3. Which files would require content-quality validation?
4. Which practices are still missing?
5. What would change for a biomedical or HPC simulation project?

> **Important caution:** The stronger example contains intentionally simple placeholder content. It demonstrates detectable structure, not a scientifically complete project.


In [ ]:
print("Weak result preview:")
print(json.dumps(weak_result, indent=2, default=str)[:2500])

print("\nStrong result preview:")
print(json.dumps(strong_result, indent=2, default=str)[:2500])


# Part 5 — Try It Yourself

Choose one of the following options.

## Option A: Assess a local repository

Replace the path below with a project that you are permitted to inspect.

## Option B: Create an improvement plan

Without running private code, use the ReproPilot categories to design a five-item reproducibility improvement plan.

## Option C: Explore domain adaptation

Select `biomedical`, `climate`, or `hpc-simulation` and consider which additional evidence should be required.


In [ ]:
# Change this path only when you have permission to inspect the repository.
PROJECT_TO_ASSESS = PROJECT_PATH
PROJECT_DOMAIN = "general"

custom_result = assess_repository(Path(PROJECT_TO_ASSESS), PROJECT_DOMAIN)
print_assessment(custom_result)


## Reflection Questions

Record your observations:

| Question | Notes |
|---|---|
| What is the strongest current practice? | |
| What is the highest-priority gap? | |
| Which finding is supported by clear evidence? | |
| Which result requires manual verification? | |
| What can be improved in one day? | |
| What requires longer-term team or infrastructure work? | |
| Which domain-specific practice is missing? | |


# Part 6 — Limitations and Critical Interpretation

ReproPilot can identify visible repository practices, but it cannot by itself prove:

- scientific validity or correctness;
- appropriate experimental design;
- absence of data leakage or bias;
- numerical stability;
- correctness across architectures or accelerators;
- successful container builds;
- meaningful test coverage;
- valid provenance for external data;
- legal or ethical compliance;
- reproducibility of nondeterministic or large-scale HPC runs.

File presence and non-empty content are useful signals, but they are not substitutes for execution, review, replication, or domain expertise.


## Questions for Further Exploration

1. Should all reproducibility categories have equal weight?
2. How should domain-specific criteria be governed and updated?
3. What evidence should be machine-verifiable?
4. How can false positives and false negatives be measured?
5. Should a repository receive a score when key evidence is private?
6. How should the tool communicate uncertainty?
7. How could ReproPilot be integrated into code review, CI, or research onboarding?


# Part 7 — Next Steps

Improve one repository practice and reassess the project.

Suggested changes:

- add exact setup and execution instructions;
- pin or lock dependencies;
- add a smoke test;
- add CI;
- document data provenance;
- provide a container or environment definition;
- add citation and licensing information;
- document random seeds and hardware/software assumptions;
- record expected outputs or validation criteria.

Then consider:

> Did the score change for the right reason, and does the repository now make a stronger reproducibility claim?


# Summary

ReproPilot demonstrates a practical pattern for responsible AI-assisted scientific software assessment:

1. **Deterministic checks establish the score.**
2. **Repository evidence makes findings inspectable.**
3. **Optional local AI explains and prioritizes evidence.**
4. **Human and domain review remain essential.**
5. **Reassessment turns recommendations into an improvement cycle.**

Repository: https://github.com/szuananwar/ai-assisted-reproducibility-bssw  
Release: https://github.com/szuananwar/ai-assisted-reproducibility-bssw/releases/tag/v1.1.0
